# 09 — Approved-drug screening for AML targets

## Goal

Prepare a deduplicated approved small-molecule library, cross it with the established AML target panel, and explicitly run baseline and advanced inference. Results are computationally prioritised drug-repurposing candidates—not experimental or clinical validation.

**Execution policy:** preparation, AML ESM-2 cache generation, and screening are disabled independently by default.


## Setup


In [ ]:
from pathlib import Path
from importlib import import_module
import json
import subprocess
import sys


def find_project_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the project checkout.")


def require_api(module_name: str, attribute: str):
    module = import_module(module_name)
    if not hasattr(module, attribute):
        raise RuntimeError(
            f"Required v2 API {module_name}.{attribute} is unavailable. "
            "Update v2-refactor and rerun the tests before enabling this cell."
        )
    return getattr(module, attribute)


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

BRANCH = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if BRANCH != "v2-refactor":
    raise RuntimeError(f"Expected v2-refactor, found {BRANCH!r}. v1-final is read-only.")

print(f"Project: {PROJECT_ROOT}")
print(f"Branch:  {BRANCH}")


In [ ]:
from config import (
    AML_DATA_DIR, ESM2_CACHE_DIR, ESM2_MODEL_NAME, MODELS_DIR,
    SCREENING_RESULTS_DIR,
)

RAW_APPROVED_DRUGS = AML_DATA_DIR / "approved_drugs_raw.csv"
AML_TARGETS = AML_DATA_DIR / "aml_targets.csv"
PREPARED_DRUGS = AML_DATA_DIR / "approved_drugs_metadata.csv"
PREPARED_TARGETS = AML_DATA_DIR / "aml_targets_prepared_v2.csv"
PAIRS_PATH = SCREENING_RESULTS_DIR / "aml_approved_drug_pairs.csv"
BASELINE_CHECKPOINT = MODELS_DIR / "selected_baseline.pt"
ADVANCED_CHECKPOINT = MODELS_DIR / "selected_advanced.pt"
AML_RESIDUE_CACHE = ESM2_CACHE_DIR / "aml_targets_esm2_t6_8m_residues.pt"
AML_RESIDUE_CACHE_METADATA = AML_RESIDUE_CACHE.with_suffix(".metadata.json")
ESM2_MODEL_REVISION = None  # pin an immutable Hugging Face commit for final runs

RUN_PREPARATION = False
RUN_AML_EMBEDDING = False
RUN_SCREENING = False


## Steps


In [ ]:
if RUN_PREPARATION:
    import pandas as pd
    from src.screening import (
        build_screening_pairs,
        prepare_aml_targets,
        prepare_approved_drugs,
    )

    raw_drugs = pd.read_csv(RAW_APPROVED_DRUGS)
    raw_targets = pd.read_csv(AML_TARGETS)
    drugs, rejected = prepare_approved_drugs(
        raw_drugs, return_rejections=True
    )
    targets = prepare_aml_targets(raw_targets)
    pairs = build_screening_pairs(drugs, targets)

    SCREENING_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    drugs.to_csv(PREPARED_DRUGS, index=False)
    targets.to_csv(PREPARED_TARGETS, index=False)
    pairs.to_csv(PAIRS_PATH, index=False)
    rejected.to_csv(SCREENING_RESULTS_DIR / "approved_drug_rejections.csv", index=False)
    print({"drugs": len(drugs), "targets": len(targets), "pairs": len(pairs), "rejected": len(rejected)})
else:
    print("Screening-input preparation skipped: RUN_PREPARATION is False.")


In [ ]:
if RUN_AML_EMBEDDING:
    import pandas as pd
    from src.preprocessing import build_esm2_residue_embedding_cache

    aml_targets = pd.read_csv(AML_TARGETS)
    residue_embedding_cache = build_esm2_residue_embedding_cache(
        dataframe=aml_targets,
        output_path=AML_RESIDUE_CACHE,
        model_name=ESM2_MODEL_NAME,
        model_revision=ESM2_MODEL_REVISION,
    )
    if not AML_RESIDUE_CACHE_METADATA.exists():
        raise FileNotFoundError(
            f"Required cache sidecar was not created: {AML_RESIDUE_CACHE_METADATA}"
        )
    print({
        "cached_targets": len(residue_embedding_cache),
        "cache_path": str(AML_RESIDUE_CACHE),
        "metadata_path": str(AML_RESIDUE_CACHE_METADATA),
    })
else:
    print("AML ESM-2 cache generation skipped: RUN_AML_EMBEDDING is False.")


In [ ]:
if RUN_SCREENING:
    import hashlib
    import pandas as pd
    from src.preprocessing import load_residue_embedding_cache
    from src.screening import run_dual_model_aml_screening

    SCREENING_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    pairs = pd.read_csv(PAIRS_PATH)
    residue_embedding_cache = load_residue_embedding_cache(
        AML_RESIDUE_CACHE
    )
    result = run_dual_model_aml_screening(
        pairs=pairs,
        baseline_checkpoint=BASELINE_CHECKPOINT,
        advanced_checkpoint=ADVANCED_CHECKPOINT,
        residue_embedding_cache=residue_embedding_cache,
        batch_size=32,
    )

    def fingerprint_file(path, chunk_size=1024 * 1024):
        path = Path(path)
        if not path.is_file():
            raise FileNotFoundError(path)
        digest = hashlib.sha256()
        with path.open("rb") as file:
            for chunk in iter(lambda: file.read(chunk_size), b""):
                digest.update(chunk)
        return {
            "path": str(path.resolve()),
            "bytes": path.stat().st_size,
            "sha256": digest.hexdigest(),
        }

    screening_provenance = {
        "runner_provenance": result.provenance,
        "input_files": {
            "approved_drugs_raw_csv": fingerprint_file(RAW_APPROVED_DRUGS),
            "aml_targets_csv": fingerprint_file(AML_TARGETS),
            "residue_cache_tensor": fingerprint_file(AML_RESIDUE_CACHE),
            "residue_cache_metadata": fingerprint_file(
                AML_RESIDUE_CACHE_METADATA
            ),
        },
    }

    result.baseline_predictions.to_csv(
        SCREENING_RESULTS_DIR / "baseline_predictions_full.csv", index=False
    )
    result.advanced_predictions.to_csv(
        SCREENING_RESULTS_DIR / "advanced_predictions_full.csv", index=False
    )
    for model_name, ranking in (
        ("baseline", result.baseline_rankings),
        ("advanced", result.advanced_rankings),
    ):
        ranking.top_10_overall.to_csv(SCREENING_RESULTS_DIR / f"{model_name}_top10.csv", index=False)
        ranking.top_20_overall.to_csv(SCREENING_RESULTS_DIR / f"{model_name}_top20.csv", index=False)
        ranking.top_5_per_target.to_csv(SCREENING_RESULTS_DIR / f"{model_name}_top5_per_target.csv", index=False)

    result.comparison.pair_comparison.to_csv(
        SCREENING_RESULTS_DIR / "pair_rank_comparison.csv", index=False
    )
    result.comparison.shared_top_drugs.to_csv(
        SCREENING_RESULTS_DIR / "shared_top_drugs.csv", index=False
    )
    result.comparison.top_pair_overlap.to_csv(
        SCREENING_RESULTS_DIR / "top_pair_overlap.csv", index=False
    )
    result.comparison.target_agreement.to_csv(
        SCREENING_RESULTS_DIR / "target_agreement.csv", index=False
    )
    (SCREENING_RESULTS_DIR / "comparison_summary.json").write_text(
        json.dumps(result.comparison.summary, indent=2, sort_keys=True, default=str),
        encoding="utf-8",
    )
    (SCREENING_RESULTS_DIR / "screening_provenance.json").write_text(
        json.dumps(
            screening_provenance,
            indent=2,
            sort_keys=True,
            default=str,
        ),
        encoding="utf-8",
    )
    print(json.dumps(screening_provenance, indent=2, default=str))
else:
    print("AML screening skipped: RUN_SCREENING is False.")


## Checks

The prepared drug table should retain source/approval metadata, canonical SMILES, and InChIKey where available; reject invalid, duplicate-structure, mixture, and graph-incompatible rows. The AML residue cache must include its strict provenance sidecar, produced with an immutable `ESM2_MODEL_REVISION` for the final run. The high-level runner accepts only compatible standard v2 checkpoints, confirms their dataset/affinity scale, and emits checkpoint/cache provenance. The saved provenance also records chunked SHA-256 and byte size for the raw approved-drug CSV, AML target CSV, cache tensor, and cache sidecar. Confirm every predicted score is finite and labelled with the dataset-specific score interpretation.

## Next steps

Use notebook 10 for the deliberately simple rank/overlap comparison. Do not add a ChEMBL/PubMed/ClinicalTrials “clinical validation” pipeline.
